In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import requests
print("✅ Дайын!")

✅ Дайын!


In [5]:
# Nuclear warheads — Wikipedia деректері
nuclear_data_real = {
    'country': ['Russia', 'USA', 'China', 'France', 'UK', 
                'Pakistan', 'India', 'Israel', 'North Korea'],
    'warheads_2023': [5889, 5244, 410, 290, 225, 170, 164, 90, 40],
    'warheads_2010': [11000, 9600, 240, 300, 225, 110, 80, 80, 0],
    'warheads_2000': [21000, 10577, 232, 348, 280, 52, 50, 75, 0],
    'latitude': [61.5, 37.1, 35.8, 46.2, 55.3, 30.3, 20.5, 31.0, 40.3],
    'longitude': [105.3, -95.7, 104.1, 2.2, -3.4, 69.3, 78.9, 34.8, 127.5],
    'nti_security_score': [34, 78, 42, 71, 73, 30, 36, 48, 10],
    'risk_level': ['High', 'Low', 'Medium', 'Low', 'Low', 
                   'High', 'Medium', 'Medium', 'Critical']
}

df_nuclear_real = pd.DataFrame(nuclear_data_real)

# Тренд есептеу
df_nuclear_real['reduction_pct'] = ((df_nuclear_real['warheads_2000'] - 
                                      df_nuclear_real['warheads_2023']) / 
                                      df_nuclear_real['warheads_2000'] * 100).round(1)

print("✅ Nuclear деректері дайын!")
print(df_nuclear_real[['country', 'warheads_2023', 'nti_security_score', 'risk_level']])

✅ Nuclear деректері дайын!
       country  warheads_2023  nti_security_score risk_level
0       Russia           5889                  34       High
1          USA           5244                  78        Low
2        China            410                  42     Medium
3       France            290                  71        Low
4           UK            225                  73        Low
5     Pakistan            170                  30       High
6        India            164                  36     Medium
7       Israel             90                  48     Medium
8  North Korea             40                  10   Critical


In [6]:
# Nuclear визуализация — нақты деректер
fig1 = px.bar(
    df_nuclear_real.sort_values('warheads_2023', ascending=True),
    x='warheads_2023',
    y='country',
    orientation='h',
    color='nti_security_score',
    color_continuous_scale='RdYlGn',
    title='☢️ Nuclear Warhead Stockpiles 2023 (FAS Data)',
    labels={'warheads_2023': 'Active Warheads', 'country': 'Country', 'nti_security_score': 'NTI Security Score'},
    text='warheads_2023'
)

fig1.update_traces(textposition='outside')
fig1.update_layout(paper_bgcolor='#0a0a0a', plot_bgcolor='#1a1a2e', font=dict(color='white'))
fig1.show()

# Тренд — 2000 vs 2023
df_trend = df_nuclear_real.melt(
    id_vars=['country'],
    value_vars=['warheads_2000', 'warheads_2010', 'warheads_2023'],
    var_name='year', value_name='warheads'
)
df_trend['year'] = df_trend['year'].str.replace('warheads_', '')

fig2 = px.line(
    df_trend,
    x='year', y='warheads', color='country',
    title='☢️ Nuclear Arsenal Trends 2000–2023',
    markers=True,
    labels={'warheads': 'Warheads', 'year': 'Year'}
)

fig2.update_layout(paper_bgcolor='#0a0a0a', plot_bgcolor='#1a1a2e', font=dict(color='white'))
fig2.show()

In [11]:
# GDELT — Global Database of Events, Language, and Tone
import requests
from datetime import datetime, timedelta

# Соңғы күннің деректері
yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y%m%d")
url = f"http://data.gdeltproject.org/events/{yesterday}.export.CSV.zip"

try:
    import zipfile, io
    r = requests.get(url, timeout=30)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    csv_name = z.namelist()[0]
    
    df_conflict_real = pd.read_csv(
        z.open(csv_name), sep='\t', header=None,
        usecols=[0,1,5,6,28,29,30],
        names=['date','year','actor1','actor2','lat','lon','country']
    ).dropna()
    
    print(f"✅ GDELT деректері: {len(df_conflict_real)} оқиға!")
    print(df_conflict_real.head())
except Exception as e:
    print(f"❌ {e}")

✅ GDELT деректері: 56112 оқиға!
         date      year actor1          actor2  lat  lon  country
1  1297567581  20250404    EDU      UNIVERSITY    2    1      3.0
2  1297567582  20250404    EDU      UNIVERSITY    7    2      7.4
3  1297567583  20250404    GOV  ADMINISTRATION    3    1      4.0
4  1297567584  20250404    HLH        HOSPITAL    4    1      2.8
5  1297567585  20250404    HLH        HOSPITAL    4    1      2.8


In [12]:
# GDELT — Full conflict analysis with English comments
r = requests.get(url, timeout=30)
z = zipfile.ZipFile(io.BytesIO(r.content))
csv_name = z.namelist()[0]

df_gdelt = pd.read_csv(
    z.open(csv_name), sep='\t', header=None,
    usecols=[1, 5, 6, 26, 27, 28, 29],
    names=['date', 'actor1', 'actor2', 'goldstein', 'num_mentions', 'lat', 'lon']
).dropna()

# Goldstein scale: negative = conflict, positive = cooperation
df_gdelt['is_conflict'] = df_gdelt['goldstein'] < 0
df_conflict_gdelt = df_gdelt[df_gdelt['is_conflict']].copy()

print(f"✅ Total events loaded: {len(df_gdelt)}")
print(f"⚔️ Conflict events: {len(df_conflict_gdelt)}")
print(f"📊 Goldstein scale — min: {df_gdelt['goldstein'].min()}, max: {df_gdelt['goldstein'].max()}")
print(df_conflict_gdelt.head())

✅ Total events loaded: 56112
⚔️ Conflict events: 0
📊 Goldstein scale — min: 10, max: 1832
Empty DataFrame
Columns: [date, actor1, actor2, goldstein, num_mentions, lat, lon, is_conflict]
Index: []


In [18]:
r = requests.get(url, timeout=30)
z = zipfile.ZipFile(io.BytesIO(r.content))
csv_name = z.namelist()[0]

df_gdelt = pd.read_csv(
    z.open(csv_name), sep='\t', header=None
)

# Convert correct columns
df_gdelt['lat'] = pd.to_numeric(df_gdelt[46], errors='coerce')
df_gdelt['lon'] = pd.to_numeric(df_gdelt[47], errors='coerce')
df_gdelt['goldstein'] = pd.to_numeric(df_gdelt[30], errors='coerce')
df_gdelt['num_mentions'] = pd.to_numeric(df_gdelt[31], errors='coerce')
df_gdelt['country'] = df_gdelt[44]
df_gdelt['actor1'] = df_gdelt[15]

# Filter
df_gdelt = df_gdelt.dropna(subset=['lat', 'lon', 'goldstein'])
df_gdelt = df_gdelt[
    (df_gdelt['lat'].between(-90, 90)) & 
    (df_gdelt['lon'].between(-180, 180))
]

df_conflict_gdelt = df_gdelt[df_gdelt['goldstein'] < 0].copy()

print(f"✅ Total events: {len(df_gdelt)}")
print(f"⚔️ Conflict events: {len(df_conflict_gdelt)}")
print(df_conflict_gdelt[['country', 'lat', 'lon', 'goldstein']].head())

✅ Total events: 41750
⚔️ Conflict events: 14524
   country      lat       lon  goldstein
12      US  30.2266  -93.2174       -2.0
15      US  31.1060  -97.6475       -2.0
18      AS -37.8167  144.9670       -5.0
24      US  31.1801  -91.8749       -5.0
27      CA  49.8833  -97.1667       -2.0


In [19]:
# GDELT Conflict Events Map — Real Data
fig3 = px.scatter_geo(
    df_conflict_gdelt.sample(min(3000, len(df_conflict_gdelt))),
    lat='lat',
    lon='lon',
    color='goldstein',
    color_continuous_scale='Reds_r',
    size='num_mentions',
    size_max=15,
    hover_name='country',
    hover_data={'goldstein': True, 'num_mentions': True},
    title='⚔️ Global Conflict Events — GDELT Real-time Data (2025)',
    labels={'goldstein': 'Conflict Intensity', 'num_mentions': 'Media Coverage'},
    projection='natural earth'
)

fig3.update_layout(
    paper_bgcolor='black',
    geo=dict(
        bgcolor='#0a0a0a',
        landcolor='#1a1a2e',
        oceancolor='#16213e',
        showland=True,
        showocean=True
    ),
    font=dict(color='white')
)

fig3.show()

In [20]:
# Top 10 countries by conflict events
top_conflict = df_conflict_gdelt.groupby('country').agg(
    conflict_events=('goldstein', 'count'),
    avg_intensity=('goldstein', 'mean'),
    total_mentions=('num_mentions', 'sum')
).reset_index().sort_values('conflict_events', ascending=False).head(10)

print("Top 10 conflict countries:")
print(top_conflict)

fig4 = px.bar(
    top_conflict.sort_values('conflict_events', ascending=True),
    x='conflict_events',
    y='country',
    orientation='h',
    color='avg_intensity',
    color_continuous_scale='Reds_r',
    title='⚔️ Top 10 Countries by Conflict Events — GDELT 2025',
    labels={
        'conflict_events': 'Number of Conflict Events',
        'country': 'Country Code',
        'avg_intensity': 'Avg Intensity'
    },
    text='conflict_events'
)

fig4.update_traces(textposition='outside')
fig4.update_layout(
    paper_bgcolor='#0a0a0a',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white')
)

fig4.show()

Top 10 conflict countries:
    country  conflict_events  avg_intensity  total_mentions
138      US             3947      -5.470231           39979
65       IR             2289      -6.446658           45689
66       IS             1241      -6.857937           15670
64       IN              728      -4.787363            4895
136      UK              676      -5.095414           11587
95       NI              550      -5.532000            2841
114      RS              437      -6.204348            3459
137      UP              328      -6.794817            2518
81       LE              293      -6.939590            5624
29       CH              247      -4.849798            2483


In [25]:
# Search available OWID disaster datasets
import requests

url = "https://api.github.com/repos/owid/owid-datasets/contents/datasets"
r = requests.get(url)
folders = r.json()

# Find disaster-related datasets
disaster_folders = [f['name'] for f in folders if 'disaster' in f['name'].lower() or 'emdat' in f['name'].lower()]
print("Available disaster datasets:")
for f in disaster_folders:
    print(f)

Available disaster datasets:
Economic losses from disasters as a share of GDP (Pielke, 2018)
Global death rates from disasters (EMDAT; UN & HYDE)
Natural disasters (EMDAT – decadal)
Natural disasters (EMDAT)
Natural disasters from 1900 to 2019 - EMDAT (2020)
Newsworthiness of disasters by disaster type and region - Eisensee and Strömberg (2007)


In [27]:
# Fix URL encoding
import urllib.parse

name = "Natural disasters (EMDAT)"
encoded = urllib.parse.quote(name)
url_emdat = f"https://raw.githubusercontent.com/owid/owid-datasets/master/datasets/{encoded}/{encoded}.csv"

try:
    df_emdat_real = pd.read_csv(url_emdat)
    print(f"✅ Loaded: {len(df_emdat_real)} records")
    print(df_emdat_real.columns.tolist())
    print(df_emdat_real.head())
except Exception as e:
    print(f"❌ {e}")

✅ Loaded: 5569 records
['Entity', 'Year', 'deaths_drought', 'injured_drought', 'affected_drought', 'homeless_drought', 'total_affected_drought', 'reconstruction_costs_drought', 'insured_damages_drought', 'total_damages_drought', 'affected_rate_per_100k_drought', 'homeless_rate_per_100k_drought', 'deaths_earthquake', 'injured_earthquake', 'affected_earthquake', 'homeless_earthquake', 'total_affected_earthquake', 'reconstruction_costs_earthquake', 'insured_damages_earthquake', 'total_damages_earthquake', 'affected_rate_per_100k_earthquake', 'homeless_rate_per_100k_earthquake', 'deaths_all_disasters', 'injured_all_disasters', 'affected_all_disasters', 'homeless_all_disasters', 'total_affected_all_disasters', 'reconstruction_costs_all_disasters', 'insured_damages_all_disasters', 'total_damages_all_disasters', 'affected_rate_per_100k_all_disasters', 'homeless_rate_per_100k_all_disasters', 'deaths_volcanic', 'injured_volcanic', 'affected_volcanic', 'homeless_volcanic', 'total_affected_volcan

In [28]:
# EM-DAT — Global deaths by disaster type (2000-2023)
df_emdat_recent = df_emdat_real[df_emdat_real['Year'] >= 2000].copy()

disaster_deaths = {
    'Earthquake': df_emdat_recent['deaths_earthquake'].sum(),
    'Flood': df_emdat_recent['deaths_flood'].sum(),
    'Storm': df_emdat_recent['deaths_storm'].sum(),
    'Drought': df_emdat_recent['deaths_drought'].sum(),
    'Wildfire': df_emdat_recent['deaths_wildfire'].sum(),
    'Landslide': df_emdat_recent['deaths_landslide'].sum(),
    'Volcanic': df_emdat_recent['deaths_volcanic'].sum(),
    'Extreme Temp': df_emdat_recent['deaths_temperature'].sum(),
}

df_disaster_deaths = pd.DataFrame({
    'disaster_type': list(disaster_deaths.keys()),
    'total_deaths': list(disaster_deaths.values())
}).dropna().sort_values('total_deaths', ascending=True)

print("✅ EM-DAT disaster deaths (2000-2023):")
print(df_disaster_deaths)

fig5 = px.bar(
    df_disaster_deaths,
    x='total_deaths',
    y='disaster_type',
    orientation='h',
    color='total_deaths',
    color_continuous_scale='Reds',
    title='🌪️ Global Deaths by Disaster Type 2000-2023 — EM-DAT Real Data',
    labels={'total_deaths': 'Total Deaths', 'disaster_type': 'Disaster Type'},
    text='total_deaths'
)

fig5.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig5.update_layout(
    paper_bgcolor='#0a0a0a',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white')
)

fig5.show()

✅ EM-DAT disaster deaths (2000-2023):
  disaster_type  total_deaths
4      Wildfire        3160.0
6      Volcanic        3202.0
5     Landslide       36680.0
3       Drought       42582.0
1         Flood      221596.0
7  Extreme Temp      344532.0
2         Storm      402970.0
0    Earthquake     1442622.0


In [29]:
# EM-DAT — Deaths trend over years
df_world = df_emdat_real[df_emdat_real['Entity'] == 'World'].copy()
df_world = df_world[df_world['Year'] >= 1990]

fig6 = px.line(
    df_world,
    x='Year',
    y=['deaths_earthquake', 'deaths_flood', 'deaths_storm', 
       'deaths_drought', 'deaths_wildfire'],
    title='📈 Global Disaster Deaths Trend 1990-2023 — EM-DAT',
    labels={'value': 'Deaths', 'variable': 'Disaster Type', 'Year': 'Year'},
    color_discrete_sequence=['#ff4444', '#ff8800', '#ffcc00', '#00ccff', '#00ff88']
)

fig6.update_layout(
    paper_bgcolor='#0a0a0a',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white'),
    legend=dict(bgcolor='rgba(0,0,0,0)')
)

fig6.show()

# EM-DAT — Economic damages
disaster_damages = {
    'Earthquake': df_emdat_recent['total_damages_earthquake'].sum(),
    'Flood': df_emdat_recent['total_damages_flood'].sum(),
    'Storm': df_emdat_recent['total_damages_storm'].sum(),
    'Drought': df_emdat_recent['total_damages_drought'].sum(),
    'Wildfire': df_emdat_recent['total_damages_wildfire'].sum(),
}

df_damages = pd.DataFrame({
    'disaster_type': list(disaster_damages.keys()),
    'total_damages_usd': list(disaster_damages.values())
}).dropna().sort_values('total_damages_usd', ascending=True)

fig7 = px.bar(
    df_damages,
    x='total_damages_usd',
    y='disaster_type',
    orientation='h',
    color='total_damages_usd',
    color_continuous_scale='Reds',
    title='💰 Economic Damages by Disaster Type 2000-2023 — EM-DAT',
    labels={'total_damages_usd': 'Total Damages (USD)', 'disaster_type': 'Disaster Type'},
    text='total_damages_usd'
)

fig7.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
fig7.update_layout(
    paper_bgcolor='#0a0a0a',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white')
)

fig7.show()